# Dataset 1 — Retail Supply Chain Sales

**Author:** Mohd Ashraf Huzairie  
**Study:** Comparative Performance Analysis of Forecasting Models for Automated Warehouse Replenishment

This concise notebook demonstrates the reproducible Dataset 1 workflow. Repeated model implementations and verbose training logs are intentionally delegated to the project package.

## 1. Setup

Place the original Excel file at the documented relative path. The dataset is not redistributed by this repository.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from warehouse_forecasting.data import load_demand_series
from warehouse_forecasting.experiment import run_experiment
from warehouse_forecasting.paper_results import table as paper_table

DATA_PATH = ROOT / 'data/dataset1/Retail-Supply-Chain-Sales-Dataset.xlsx'
DATASET = 'retail_supply_chain'
RUN_TRAINING = False
print(f'Repository: {ROOT}')
print(f'Dataset available: {DATA_PATH.exists()}')

## 2. Clean and aggregate demand

The shared loader parses dates, converts the target to numeric values, aggregates demand daily, interpolates missing dates, and clips impossible negative demand. Scaling is fitted separately inside each training fold to prevent leakage.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Download Dataset 1 and place it at {DATA_PATH.relative_to(ROOT)}')
series = load_demand_series(DATA_PATH, DATASET)
display(series.describe().to_frame('daily_quantity'))
print(f'Date range: {series.index.min().date()} to {series.index.max().date()}')
print(f'Missing values after cleaning: {series.isna().sum()}')

## 3. Compact exploratory analysis

The daily series is shown together with a 30-day rolling mean. This keeps the public notebook focused on demand trend and seasonality rather than unrelated business dashboards.

In [ ]:
ax = series.plot(figsize=(12, 4), alpha=.35, label='Daily quantity')
series.rolling(30, min_periods=1).mean().plot(ax=ax, linewidth=2, label='30-day mean')
ax.set(title='Dataset 1 daily demand', xlabel='Date', ylabel='Quantity')
ax.legend(); plt.tight_layout()

## 4. Leakage-safe model comparison

The showcase runs two lightweight nonlinear baselines using expanding-window cross-validation. The full CLI supports all eight study models. Deep-learning experiments are opt-in because they require TensorFlow and substantial compute.

In [ ]:
if RUN_TRAINING:
    summary = run_experiment(series, ['mlp', 'rbf'], ROOT / 'artifacts/dataset1', lookback=30, n_splits=5)
    display(summary.sort_values('rmse'))
else:
    print('Training skipped. Set RUN_TRAINING = True to run MLP and RBF.')

## 5. Results reported in the paper

These values are a transcription of the paper tables, not results generated by this notebook session.

In [ ]:
published = paper_table().query("dataset == 'dataset_1'").pivot(index='model', columns='metric', values='value')
display(published[['mse', 'rmse', 'mae', 'r2', 'mape']].sort_values('rmse'))

## Interpretation

Dataset 1 produced strong results for MLP and LSTM in the reported tables. LSTM had the lower MAE and MAPE, while MLP had the lowest MSE and RMSE and highest R². These metric-specific differences should be reported rather than describing one model as universally best.